<a href="https://colab.research.google.com/github/csmasuda/santander-dev-week-2025/blob/csmasuda/SantanderDevWeek2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Santander Dev Week 2025 (ETL com Python)

# Informações Iniciais

Como não pretendia "gastar" o periodo de teste do api para o projeto, decidi fazer um desafio a mim mesmo (até seguindo o que o Venilton sugeriu) colocando na prática o que geralmente faço, mas utilizando outras plataformas.

Sou iniciante em Python, então relamente tem muito código que pode ser aprimorado ainda

**Contexto:** Você é um cientista de dados no Santander e recebeu a tarefa de envolver seus clientes de maneira mais personalizada. *<font color="red"> O objetivo é padronizar os dados da base e vincular às suas preferências de investimento</font>*.

**Condições do Problema:**

1. Você recebeu uma planilha simples, em formato CSV ('SDW202<font color="red">**5**</font>.csv'), com *<font color="red">uma lista de clientes do banco obtida de diferentes origens, mas todos da cidade de São Paulo</font>*:
  ```
  UserID, User_Name, User_Birth, User_Genre, User_Phone,User_Acept_SMS
1, User1, 21/05/2010,Masculino,5511999999999,No
2, User2, 05/10/1957,Female,+55(11)99999-9999,Yes
3,User3,29/11/1998,Outros,99999-9999,Yes
4,User4,01/01/1990,Male,999999999,yes
5,User5,15/04/2008,feminino,99999-9999,1
6,User6,06/07/1984,Mulher,123456,Sim
7,user7,04/08/2000,Male,123456789,True
  ```

2. Seu trabalho é extrair os dados de clientes em csv e slava-lo em json **<font color="blue">[EXTRACT]</font>**.

3. Em seguida, deve tratar os dados a fim de padronizá-los para que possam ser utilizados para análise posterior. **<font color="blue">[TRANSFORM]</font>**

4. Ainda nesta etapa, deve-se criar uma mensagem personalizada para cada usuário, a depender de suas preferencias de investimento. Para isso deve relacionar a tabela de cadastro com a de preferencias, além de avaliar se estão aptos a receberem SMS (optin), se é maior de idade e se o telefone informado é válido.
  ```
UserID,User_Interest
1,CDB
1,Fundo DI
1,Debendures
4,Tesouro Direto
4,CDB
6,Previdencia Privada
2,Tesouro Direto
3,CDB
7,Renda Variavel
9,CDB
6,Tesouro Direto
2,Fundo Multimercado
5,CDB
3,Renda Variavel

  ```
5. Por fim, deve se salvar as mensagens em uma lista dentro de cadastro para os respectivos clientes **<font color="blue">[LOAD]</font>**.


-----------------------------------------------------


## **E**xtract

Extraia a lista de IDs de usuário a partir do arquivo CSV. Para cada ID, faça uma requisição GET para obter os dados do usuário correspondente.

In [102]:
import pandas as pd

data = pd.read_csv('SDW2025.csv')

df1=pd.DataFrame(data)
print(df1)

data=pd.read_csv('SDW2025_preferencia.csv')

df2=pd.DataFrame(data)
print(df2)

   UserID  User_Name   User_Birth  User_Genre         User_Phone  \
0       1      User1   21/05/2010   Masculino      5511999999999   
1       2      User2   05/10/1957      Female  +55(11)99999-9999   
2       3      User3   29/11/1998      Outros         99999-9999   
3       4      User4   01/01/1990        Male          999999999   
4       5      User5   15/04/2008    feminino         99999-9999   
5       6      User6   06/07/1984      Mulher             123456   
6       7      user7   04/08/2000        Male          123456789   

  User_Acept_SMS  
0             No  
1            Yes  
2            Yes  
3            yes  
4              1  
5            sim  
6           True  
    UserID        User_Interest
0        1                  CDB
1        1             Fundo DI
2        1           Debendures
3        4       Tesouro Direto
4        4                  CDB
5        6  Previdencia Privada
6        2       Tesouro Direto
7        3                  CDB
8        7     

In [103]:
import csv
import json

#Criar json a partir do arquivo csv

def csv_to_json(arquivo,SDW2025):
  dados=[]

  with open(arquivo, mode='r',encoding='utf-8') as csv_file:
    csv_reader = csv.DictReader(csv_file)
    #json_data = json

    for linha in csv_reader:
      dados.append(linha)

      with open(SDW2025, mode='w', encoding='utf-8') as json_file:
        json.dump(dados, json_file, indent=4)

  return dados

arquivo = 'SDW2025.csv'
SDW2025 = 'SDW2025.json'
csv_to_json(arquivo,SDW2025)

with open(SDW2025, mode='r',encoding='utf-8') as json_file:
  impres=json.load(json_file)

print(json.dumps(impres, indent=2))

[
  {
    "\ufeffUserID": "1",
    " User_Name": " User1",
    " User_Birth": " 21/05/2010",
    " User_Genre": "Masculino",
    " User_Phone": "5511999999999",
    "User_Acept_SMS": "No"
  },
  {
    "\ufeffUserID": "2",
    " User_Name": " User2",
    " User_Birth": " 05/10/1957",
    " User_Genre": "Female",
    " User_Phone": "+55(11)99999-9999",
    "User_Acept_SMS": "Yes"
  },
  {
    "\ufeffUserID": "3",
    " User_Name": "User3",
    " User_Birth": "29/11/1998",
    " User_Genre": "Outros",
    " User_Phone": "99999-9999",
    "User_Acept_SMS": "Yes"
  },
  {
    "\ufeffUserID": "4",
    " User_Name": "User4",
    " User_Birth": "01/01/1990",
    " User_Genre": "Male",
    " User_Phone": "999999999",
    "User_Acept_SMS": "yes"
  },
  {
    "\ufeffUserID": "5",
    " User_Name": "User5",
    " User_Birth": "15/04/2008",
    " User_Genre": "feminino",
    " User_Phone": "99999-9999",
    "User_Acept_SMS": "1"
  },
  {
    "\ufeffUserID": "6",
    " User_Name": "User6",
    " Use

## **T**ransform

Utilize a API do OpenAI GPT-4 para gerar uma mensagem de marketing personalizada para cada usuário.

In [104]:
from ast import IsNot
# Padronizar e Corrigir dados

from numpy import number
from datetime import datetime

import pandas as pd
import csv
import json

#Duplicar CSV

clone='SDW2025_2.csv'

with open(arquivo, 'rb') as original:
    with open(clone, 'wb') as alterado:
          alterado.write(original.read())

df3=pd.read_csv('SDW2025_2.csv', encoding='utf-8-sig')

# Limpeza dos nomes da colunas (Sugerido pelo Gemini)
df3.columns = df3.columns.str.strip().str.replace('\ufeff', '')

# Identificação das colunas que serão impactados pelo strip (Sugerido pelo Gemini)
string_columns_to_strip = ['User_Name', 'User_Birth', 'User_Genre', 'User_Phone', 'User_Acept_SMS']

# Usar o comando strip para cada coluna (Sugerido pelo Gemini)
for col in string_columns_to_strip:
    if col in df3.columns and df3[col].dtype == 'object': # Check if column exists and is of object type (usually strings)
        df3[col] = df3[col].str.strip()
        df3[col] = df3[col].str.capitalize() #Colocar a primeira letra em maiuscula em todos os reusltados

#Alterar os dados de gênero para padronização
df3['User_Genre'] = df3['User_Genre'].replace({
        'Male': 'Masculino',
        'Female': 'Feminino',
        'Mulher': 'Feminino'
})

df3['pronome'] = df3['User_Genre']
df3['pronome'] = df3['pronome'].replace({
        'Masculino': 'Sr.',
        'Feminino': 'Sra.',
        'Outros': ''
})

#Limpar simbolos do telefone

df3['User_Phone'] = df3['User_Phone'].astype(str)

df3['User_Phone'] = df3['User_Phone'].str.replace('5511','')
df3['User_Phone'] = df3['User_Phone'].str.replace('+55(11)','')
df3['User_Phone'] = df3['User_Phone'].str.replace('-','')

df3['User_Phone'] = df3['User_Phone'].astype(str)

#Tentativa de analise de telefone valido com for e if, mas não consegui fazer salvar o valor dentro da lista

# for tel in df3['User_Phone']:
#   #tel=df3['User_Phone'].str.len()

#   comp=len(tel)

#   if comp<9:
#     tel=''
#   else:
#     tel=tel
#   print(tel,comp)

#Após pesquisar, foi sugerido esta solução beeem mais simples:

checagem = df3['User_Phone'].str.len() < 9
df3.loc[checagem, 'User_Phone'] = ''


#Padronizar o opt-ins de recebimento de SMS
df3['User_Acept_SMS'] = df3['User_Acept_SMS'].replace({
        'No': 'False',
        'Yes': 'True',
        '1': 'True',
        'Sim': 'True'
})

#hoje=datetime.today().strftime('%d/%m/%Y')
hoje = pd.Timestamp(datetime.today())
df3['User_Birth']=pd.to_datetime(df3['User_Birth'],format='%d/%m/%Y')
#df3['User_Birth']=datetime.strptime(df3['User_Birth'],'%d/%m/%Y').date()
df3['hoje']=hoje
#idade=
df3['idade']=(df3['hoje']-df3['User_Birth']).dt.days/365.25
#df3['idade'	]=df3['dias']/365
df3['idade']=df3['idade'].astype(int)

#definir quem é maior de idade
maioridade=df3['idade']>=18
df3.loc[maioridade,'idade']=1

#indicar quem é menor de idade
menor=df3['idade']!= 1
df3.loc[menor,'idade']=0

df3=df3.drop(columns=['hoje'])

#df3['maior']=idade


#Atualizar os dados dentro do clone csv
df3.to_csv('SDW2025_2.csv', index=False)

#Criar o json do clone (somente para comparação com o original)
def csv_to_json(clone, json_clone):
    data = []
    with open(clone, mode='r', encoding='utf-8-sig') as csv_file:
        csv_reader = csv.DictReader(csv_file)

        for row in csv_reader:
            # Limpeza das linhas (Sugerido pelo Gemini)
            cleaned_row = {k.strip().replace('\ufeff', ''): v.strip() for k, v in row.items()}
            data.append(cleaned_row)

    with open(json_clone, mode='w', encoding='utf-8') as json_file:
        json.dump(data, json_file, indent=4)
    return data

SDW2025f = 'SDW2025f.json'

csv_to_json(clone,SDW2025f)

with open(SDW2025f, mode='r',encoding='utf-8') as json_file:
  impres=json.load(json_file)

print(json.dumps(impres, indent=2))

[
  {
    "UserID": "1",
    "User_Name": "User1",
    "User_Birth": "2010-05-21",
    "User_Genre": "Masculino",
    "User_Phone": "999999999",
    "User_Acept_SMS": "False",
    "pronome": "Sr.",
    "idade": "0"
  },
  {
    "UserID": "2",
    "User_Name": "User2",
    "User_Birth": "1957-10-05",
    "User_Genre": "Feminino",
    "User_Phone": "999999999",
    "User_Acept_SMS": "True",
    "pronome": "Sra.",
    "idade": "1"
  },
  {
    "UserID": "3",
    "User_Name": "User3",
    "User_Birth": "1998-11-29",
    "User_Genre": "Outros",
    "User_Phone": "999999999",
    "User_Acept_SMS": "True",
    "pronome": "",
    "idade": "1"
  },
  {
    "UserID": "4",
    "User_Name": "User4",
    "User_Birth": "1990-01-01",
    "User_Genre": "Masculino",
    "User_Phone": "999999999",
    "User_Acept_SMS": "True",
    "pronome": "Sr.",
    "idade": "1"
  },
  {
    "UserID": "5",
    "User_Name": "User5",
    "User_Birth": "2008-04-15",
    "User_Genre": "Feminino",
    "User_Phone": "99999

In [105]:
import math

df4=pd.read_csv('SDW2025_2.csv')
df5=pd.read_csv('SDW2025_preferencia.csv')

df5.columns = df5.columns.str.strip()

df6=df4.merge(df5,on='UserID')
#colum_to_replace=['Pronome']
#df6['pronome'] = df6['pronome'].fillna("tirar")

#print(df6)

# df6.to_csv('SDW2025_3.csv', index=False)
# df7=pd.read_csv('SDW2025_3.csv')
# #print(df7)
df6['contato']='none'

for index, line in df6.iterrows():
  #print(line['User_Name'])

  if math.isnan(line['User_Phone']) or line['idade']==0 or line['User_Acept_SMS']==False:
    pass
  else:
    tratamento=str(line['pronome'])
    if tratamento.startswith("Sr"):
      pass
    else:
      tratamento=''
    if line['User_Interest']=='Debendures' or line['User_Interest']=='Renda Variavel':
      mensagem=f'Olá {tratamento} {line['User_Name']}, acesse a nossa corretora!'
      df6.loc[index,'contato'] = mensagem
    else:
      mensagem=f'Olá {tratamento} {line['User_Name']}, veja as nossas novidades em {line['User_Interest']}!'
      df6.loc[index,'contato'] = mensagem
    print(mensagem)

print (df6)

df6.to_csv('SDW2025_3.csv', index=False)

  # if [line,'idade']==0 and [line,'User_Acept_SMS']==True and [line,'User_Phone']!='':
  #   mensagem=f'Olá, 1{line["pronome"]} {line["User_Name"]}, {line["User_Interest"]}'
  #   print(mensagem)
  # else:
  #   if [line,'pronome']=='tirar':
  #     mensagem=f'Olá, 2 {line["pronome"]} {line["User_Name"]}'
  #   else:
  #     mensagem=f'Olá, 3 {line["User_Name"]}'
  #   print(mensagem)


# def criar(user):

#   #pronome=user['pronome']
#   #nome=user['User_Name']
#   #pref=user['User_Interest']

#   # if user['idade']==1 and user['User_Acept_SMS']==True and user['User_Phone']!='':
#   #   mensagem=f'Olá, {pronome} {nome}'

#   return mensagem

# for row in enumerate(df6):
#   #criar(user)

#   pronome=row[1]
#   nome=row['User_Name']
#   pref=row['User_Interest']

#   if row['idade']==1 and row['User_Acept_SMS']==True and row['User_Phone']!='':
#     mensagem=f'Olá, {pronome} {nome}'

# print (df6)


Olá Sra. User2, veja as nossas novidades em Tesouro Direto!
Olá Sra. User2, veja as nossas novidades em Fundo Multimercado!
Olá  User3, veja as nossas novidades em CDB!
Olá  User3, acesse a nossa corretora!
Olá Sr. User4, veja as nossas novidades em Tesouro Direto!
Olá Sr. User4, veja as nossas novidades em CDB!
Olá Sr. User7, acesse a nossa corretora!
    UserID User_Name  User_Birth User_Genre   User_Phone  User_Acept_SMS  \
0        1     User1  2010-05-21  Masculino  999999999.0           False   
1        1     User1  2010-05-21  Masculino  999999999.0           False   
2        1     User1  2010-05-21  Masculino  999999999.0           False   
3        2     User2  1957-10-05   Feminino  999999999.0            True   
4        2     User2  1957-10-05   Feminino  999999999.0            True   
5        3     User3  1998-11-29     Outros  999999999.0            True   
6        3     User3  1998-11-29     Outros  999999999.0            True   
7        4     User4  1990-01-01  Mas

## **L**oad

Atualize a lista de "news" de cada usuário na API com a nova mensagem gerada.

In [114]:
data = pd.read_csv('SDW2025.csv')

df1=pd.DataFrame(data)
df1['mensagem']=""

print(df1)

data1=pd.read_csv('SDW2025_3.csv')

df8=pd.DataFrame(data1)

#print(df8)

for index,line in df8.iterrows():
  if line['contato']!='none':
    texto = line['contato']
    for index2,line2 in df1.iterrows():
      if line['UserID']==line2['UserID']:

        novo_item = [{'SMS':texto}]
        valor_atual = df1.loc[index2, 'mensagem']
        if not isinstance(valor_atual, list) or pd.isna(valor_atual):
            valor_atual = []
            df1.loc[index2, 'mensagem'] = valor_atual + novo_item
        else:
            valor_atual.append(novo_item)

        # objeto=df1.loc[index2,'mensagem']
        # objeto.append({'SMS':texto})
        # df1.loc[index2,'mensagem']=objeto
      else:
        pass

df1.to_csv('SDW2025.csv', index=False)

atual='SDW2025.csv'
novojson='SDW2025f2.json'

csv_to_json(atual,novojson)

with open(novojson, mode='r',encoding='utf-8') as json_file:
  impres=json.load(json_file)

print(json.dumps(impres, indent=2))

   UserID  User_Name   User_Birth  User_Genre         User_Phone  \
0       1      User1   21/05/2010   Masculino      5511999999999   
1       2      User2   05/10/1957      Female  +55(11)99999-9999   
2       3      User3   29/11/1998      Outros         99999-9999   
3       4      User4   01/01/1990        Male          999999999   
4       5      User5   15/04/2008    feminino         99999-9999   
5       6      User6   06/07/1984      Mulher             123456   
6       7      user7   04/08/2000        Male          123456789   

  User_Acept_SMS mensagem  
0             No           
1            Yes           
2            Yes           
3            yes           
4              1           
5            sim           
6           True           
[
  {
    "UserID": "1",
    "User_Name": "User1",
    "User_Birth": "21/05/2010",
    "User_Genre": "Masculino",
    "User_Phone": "5511999999999",
    "User_Acept_SMS": "No",
    "mensagem": ""
  },
  {
    "UserID": "2",
    "Us

# Rascunho (OLD)

In [ ]:
# Utilize sua própria URL se quiser ;)
# Repositório da API: https://github.com/digitalinnovationone/santander-dev-week-2023-api
#sdw2023_api_url = 'https://sdw-2023-prd.up.railway.app'

In [ ]:
#Export from API

# import requests
# import json

# def get_user(id):
#   response = requests.get(f'{sdw2023_api_url}/users/{id}')
#   return response.json() if response.status_code == 200 else None

# users = [user for id in user_ids if (user := get_user(id)) is not None]
# print(json.dumps(users, indent=2))

[
  {
    "id": 4,
    "name": "Pyterson",
    "account": {
      "id": 7,
      "number": "00001-1",
      "agency": "0001",
      "balance": 0.0,
      "limit": 500.0
    },
    "card": {
      "id": 4,
      "number": "**** **** **** 1111",
      "limit": 1000.0
    },
    "features": [],
    "news": [
      {
        "id": 9,
        "icon": "https://digitalinnovationone.github.io/santander-dev-week-2023-api/icons/credit.svg",
        "description": "Pyterson, invista hoje para garantir um futuro seguro e pr\u00f3spero. Seu futuro agradece!"
      }
    ]
  },
  {
    "id": 5,
    "name": "Pip",
    "account": {
      "id": 8,
      "number": "00002-2",
      "agency": "0001",
      "balance": 0.0,
      "limit": 500.0
    },
    "card": {
      "id": 5,
      "number": "**** **** **** 2222",
      "limit": 1000.0
    },
    "features": [],
    "news": [
      {
        "id": 10,
        "icon": "https://digitalinnovationone.github.io/santander-dev-week-2023-api/icons/credit.svg",


In [ ]:
# !pip install openai

In [ ]:
# Documentação Oficial da API OpenAI: https://platform.openai.com/docs/api-reference/introduction
# Informações sobre o Período Gratuito: https://help.openai.com/en/articles/4936830

# Para gerar uma API Key:
# 1. Crie uma conta na OpenAI
# 2. Acesse a seção "API Keys"
# 3. Clique em "Create API Key"
# Link direto: https://platform.openai.com/account/api-keys

# Substitua o texto TODO por sua API Key da OpenAI, ela será salva como uma variável de ambiente.
#openai_api_key = 'TODO'

In [ ]:
# import openai

# openai.api_key = openai_api_key

# def generate_ai_news(user):
#   completion = openai.ChatCompletion.create(
#     model="gpt-4",
#     messages=[
#       {
#           "role": "system",
#           "content": "Você é um especialista em markting bancário."
#       },
#       {
#           "role": "user",
#           "content": f"Crie uma mensagem para {user['name']} sobre a importância dos investimentos (máximo de 100 caracteres)"
#       }
#     ]
#   )
#   return completion.choices[0].message.content.strip('\"')

# for user in users:
#   news = generate_ai_news(user)
#   print(news)
#   user['news'].append({
#       "icon": "https://digitalinnovationone.github.io/santander-dev-week-2023-api/icons/credit.svg",
#       "description": news
#   })

Pyterson, invista para fazer seu dinheiro crescer. Seu futuro financeiro depende disso!
Pip, investir é o caminho para multiplicar seu dinheiro. Vamos fortalecer seu futuro financeiro!
Pep, investimentos são a chave para o futuro financeiro. Cresça seu dinheiro, não apenas o guarde!


In [ ]:
# Tentativa de export

# import pandas as pd
# import csv
# import json

# #Duplicar CSV

# clone='SDW2025_2.csv'

# with open(arquivo, 'rb') as original:
#     with open(clone, 'wb') as alterado:
#           alterado.write(original.read())

# # Read the cloned CSV, handling potential BOM and ensuring consistent encoding
# df3=pd.read_csv('SDW2025_2.csv', encoding='utf-8-sig')

# # Clean column names by stripping whitespace and removing BOM if present
# df3.columns = df3.columns.str.strip().str.replace('\ufeff', '')

# # Identify string columns that might need stripping of values
# string_columns_to_strip = ['User_Name', 'User_Birth', 'User_Genre', 'User_Phone', 'User_Acept_SMS']

# # Apply .str.strip() to relevant string columns to clean their values
# for col in string_columns_to_strip:
#     if col in df3.columns and df3[col].dtype == 'object': # Check if column exists and is of object type (usually strings)
#         df3[col] = df3[col].str.strip()

# print(df3)

# # The csv_to_json function from a previous cell is needed here if it's not global
# def csv_to_json(file_path, json_file_path):
#     data = []
#     with open(file_path, mode='r', encoding='utf-8-sig') as csv_file:
#         csv_reader = csv.DictReader(csv_file)
#         for row in csv_reader:
#             # Strip keys and values from the CSV data during conversion
#             cleaned_row = {k.strip().replace('\ufeff', ''): v.strip() for k, v in row.items()}
#             data.append(cleaned_row)

#     with open(json_file_path, mode='w', encoding='utf-8') as json_file:
#         json.dump(data, json_file, indent=4)
#     return data

# SDW2025f = 'SDW2025f.json'
# # Now csv_to_json uses the cleaned clone CSV
# csv_to_json(clone,SDW2025f)

# with open(SDW2025f, mode='r',encoding='utf-8') as json_file:
#   impres=json.load(json_file)

# print(json.dumps(impres, indent=2))

   UserID User_Name  User_Birth User_Genre         User_Phone User_Acept_SMS
0       1     User1  21/05/2010  Masculino      5511999999999             No
1       2     User2  05/10/1957     Female  +55(11)99999-9999            Yes
2       3     User3  29/11/1998     Outros         99999-9999            Yes
3       4     User4  01/01/1990       Male          999999999            yes
4       5     User5  15/04/2008   feminino         99999-9999              1
5       6     User6  06/07/1984     Mulher             123456            sim
6       7     user7  04/08/2000       Male          123456789           True
[
  {
    "UserID": "1",
    "User_Name": "User1",
    "User_Birth": "21/05/2010",
    "User_Genre": "Masculino",
    "User_Phone": "5511999999999",
    "User_Acept_SMS": "No"
  },
  {
    "UserID": "2",
    "User_Name": "User2",
    "User_Birth": "05/10/1957",
    "User_Genre": "Female",
    "User_Phone": "+55(11)99999-9999",
    "User_Acept_SMS": "Yes"
  },
  {
    "UserID": "3",

In [ ]:
# Tentativa de correção de dados

# #Corrigir Dados
# SDW2025 = 'SDW2025.json'
# def fix(User):
#   NewBirth=User["User_Birth"].strip()
#   return True


# with open(SDW2025, mode='r', encoding='utf-8') as original:
#       dados = json.load(original)
#       print(f'json{dados}')

#       #dados['User_Genre']=1


#       #for User_Genre in dados:
#         #if "valor" in User_Genre:
#           #User_Genre["valor"]="Alterado"
#           #User_Birth["valor"]=User_Birth["valor"].strip()

# for user in dados:
#   fix(user)
#   with open(SDW2025, 'w', encoding='utf-8') as atual:
#         json.dump(dados, user, indent=4)

# with open(SDW2025, mode='r', encoding='utf-8') as arquivo:
#       dados = json.load(arquivo)
#       print(f'json{dados}')

#        # dados=  json.load(SDW2025)

# #print(json.dumps(dados, indent=2))




json[{'\ufeffUserID': '1', ' User_Name': ' User1', ' User_Birth': ' 21/05/2010', ' User_Genre': 'Masculino', ' User_Phone': '5511999999999', 'User_Acept_SMS': 'No'}, {'\ufeffUserID': '2', ' User_Name': ' User2', ' User_Birth': ' 05/10/1957', ' User_Genre': 'Female', ' User_Phone': '+55(11)99999-9999', 'User_Acept_SMS': 'Yes'}, {'\ufeffUserID': '3', ' User_Name': 'User3', ' User_Birth': '29/11/1998', ' User_Genre': 'Outros', ' User_Phone': '99999-9999', 'User_Acept_SMS': 'Yes'}, {'\ufeffUserID': '4', ' User_Name': 'User4', ' User_Birth': '01/01/1990', ' User_Genre': 'Male', ' User_Phone': '999999999', 'User_Acept_SMS': 'yes'}, {'\ufeffUserID': '5', ' User_Name': 'User5', ' User_Birth': '15/04/2008', ' User_Genre': 'feminino', ' User_Phone': '99999-9999', 'User_Acept_SMS': '1'}, {'\ufeffUserID': '6', ' User_Name': 'User6', ' User_Birth': '06/07/1984', ' User_Genre': 'Mulher', ' User_Phone': '123456', 'User_Acept_SMS': 'sim'}, {'\ufeffUserID': '7', ' User_Name': 'user7', ' User_Birth': '0

KeyError: 'User_Birth'

In [ ]:
# # Tentativa de Inserir as preferência dos clientes no arquivo Json

# def gerar_preferencia(UserID):

#   for UserID in SDW2025['\ufeffUserID']:
#     for UserID in df2:
#       if UserID == UserID:
#         preferencia=df2['User_Interest']
#         print(preferencia)
#         UserID['preferencia'].append(preferencia)
#         return True
#       else:
#         return False



In [ ]:
# def update_user(user):
#   response = requests.put(f"{sdw2023_api_url}/users/{user['id']}", json=user)
#   return True if response.status_code == 200 else False

# for user in users:
#   success = update_user(user)
#   print(f"User {user['name']} updated? {success}!")

User Pyterson updated? True!
User Pip updated? True!
User Pep updated? True!
